In [ ]:
import os
import sqlite3
from datetime import UTC, datetime, timedelta
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from agent.helpers.normalize import normalize

load_dotenv()

In [ ]:
@tool
async def get_collected_logs(container: str, hours: int = 24, limit: int = 100) -> str:
    """List recent lines from the agent's own collected log store for a watched
    container, covering the last `hours` (default 24, capped at 7 days). Only
    covers containers on the watch list."""
    hours = max(1, min(hours, 7 * 24))
    limit = max(1, min(limit, 500))
    rel_path = os.environ.get("DATABASE_URL", "sqlite:///data/agent.db").replace("sqlite:///", "")
    db_path = Path(find_dotenv()).parent / rel_path  # DB path is relative to repo root, not CWD
    conn = sqlite3.connect(db_path)
    try:
        watched = [row[0] for row in conn.execute("SELECT name FROM watched_containers").fetchall()]
    finally:
        conn.close()
    needle = normalize(container)
    target = next((name for name in watched if needle in normalize(name)), None)
    if target is None:
        return f"'{container}' is not a watched container; the collected log store only covers watched containers."
    log_root = Path(db_path).parent / "logs" / target
    since_day = (datetime.now(UTC) - timedelta(hours=hours)).date().isoformat()
    lines = []
    if log_root.exists():
        for day_file in sorted(log_root.glob("*.log")):
            if day_file.stem >= since_day:
                lines.extend(l for l in day_file.read_text(encoding="utf-8").splitlines() if l.strip())
    lines = lines[-limit:]
    if not lines:
        return f"No collected log lines for '{target}' in the last {hours}h."
    return "\n".join(lines)[:4000]

In [ ]:
llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ.get("LLM_API_KEY"),
    temperature=0,
).bind_tools([get_collected_logs])

In [ ]:
SYSTEM = "You are an infrastructure-operations assistant. Use the get_collected_logs tool when relevant, then answer concisely."

transcript: list = []


async def ask(question: str) -> str:
    """Ask one question. Makes at most one get_collected_logs call, then returns the model's answer."""
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    reply = await llm.ainvoke(messages)
    messages.append(reply)
    if reply.tool_calls:
        call = reply.tool_calls[0]
        result = await get_collected_logs.ainvoke(call["args"] or {})
        messages.append(ToolMessage(tool_call_id=call["id"], content=str(result)[:4000]))
        reply = await llm.ainvoke(messages)
        messages.append(reply)
    transcript[:] = messages
    return reply.content

In [ ]:
print(await ask("What's in the collected log store for api?"))

In [ ]:
def show_transcript(transcript: list) -> None:
    """Print each message with its tool calls, token usage, and reasoning."""
    for m in transcript:
        label = type(m).__name__
        body = m.content if not isinstance(m, ToolMessage) else f"[tool] {m.content}"
        if not body:
            body = "(tool call only)" if getattr(m, "tool_calls", None) else "(empty)"
        print(f"{label}: {body.replace(chr(10), chr(10) + '  ')}")

        reasoning = m.additional_kwargs.get("reasoning_content")
        if reasoning:
            print(f"  reasoning: {reasoning}")

        for call in getattr(m, "tool_calls", None) or []:
            print(f"  tool_call: {call['name']}({call['args']})")

        usage = getattr(m, "usage_metadata", None)
        if usage:
            print(f"  tokens: in={usage['input_tokens']} out={usage['output_tokens']} total={usage['total_tokens']}")

In [ ]:
show_transcript(transcript)